# ASR Training on Google Colab with Parquet (Drive)

## 📋 Setup Instructions:
1. **Runtime -> Change runtime type -> GPU (T4)**
2. **Access shared data folder** (see below)
3. Run all cells in order
4. Checkpoints will be saved to Google Drive: `/content/drive/MyDrive/asr_checkpoints/`

## 📁 Accessing Shared Data Folder:

**Shared folder link**: https://drive.google.com/drive/folders/17iqsD7J2xi9_YLQGsqopR8XX-nVFN2y3?usp=sharing

### Steps to access:
1. Open the link above in your browser
2. Right-click on the folder → **"Add shortcut to Drive"**
3. Choose **"My Drive"** as the location
4. After mounting Drive in Colab (Cell 2), the folder will be accessible
5. Run **Cell 6a** to auto-detect the folder path
6. Update **BASE_DIR** in **Cell 7** with the detected path

## 🔄 Pipeline:
Mount Drive → Install deps → Find parquet files → Load datasets → Export CSV → Import to DB → Train → Save to Drive



In [ ]:
# 1) Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


In [ ]:
# 2) Mount Google Drive
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    print("Not in Colab runtime; skipping Drive mount.")
    IN_COLAB = False


In [ ]:
# 3) Clone project from GitHub to /content/ai2text
import os
import subprocess

repo_url = "https://github.com/congkx123789/AI2Text-_new2.git"
target_dir = "/content/ai2text"

if os.path.isdir(target_dir):
    print("✓ Project already exists at", target_dir)
    print("  If you want to re-clone, delete the directory first")
else:
    print("📥 Cloning repository from", repo_url)
    result = subprocess.run(
        ['git', 'clone', repo_url, 'ai2text'],
        cwd='/content',
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Project cloned to", target_dir)
    else:
        print("❌ Clone failed:", result.stderr)

# Verify project structure
if os.path.isdir(target_dir):
    required_dirs = ['training', 'models', 'configs', 'database', 'preprocessing']
    missing = [d for d in required_dirs if not os.path.isdir(os.path.join(target_dir, d))]
    if missing:
        print("⚠️ Missing directories:", missing)
    else:
        print("✓ Project structure verified")


In [ ]:
# 4) Install dependencies optimized for T4 GPU (Colab)
import subprocess
import sys
import os

print("=" * 60)
print("📦 Installing dependencies for T4 GPU")
print("=" * 60)

# Step 1: Upgrade pip and build tools
print("\n1. Upgrading pip, setuptools, wheel...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], 
               check=False, capture_output=True)

# Step 2: Detect CUDA version and install compatible PyTorch
print("\n2. Detecting CUDA version...")
try:
    import torch
    if torch.cuda.is_available():
        cuda_version = torch.version.cuda
        print(f"   ✓ CUDA {cuda_version} detected")
        print(f"   ✓ PyTorch {torch.__version__} already installed")
        print(f"   ✓ GPU: {torch.cuda.get_device_name(0)}")
    else:
        raise ImportError("PyTorch not found or CUDA not available")
except (ImportError, AttributeError):
    # Colab usually has CUDA 11.8 or 12.1, try both
    print("   PyTorch not found, installing compatible version...")
    # Try CUDA 12.1 first (newer Colab instances)
    print("   Trying CUDA 12.1 (cu121)...")
    result = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
        'torch==2.1.0', 'torchaudio==2.1.0'
    ], check=False, capture_output=True, timeout=300)
    
    # Verify installation
    try:
        import torch
        if torch.cuda.is_available():
            print(f"   ✓ PyTorch installed with CUDA 12.1")
        else:
            # Try CUDA 11.8 as fallback
            print("   CUDA 12.1 not working, trying CUDA 11.8...")
            subprocess.run([
                sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchaudio'
            ], check=False)
            subprocess.run([
                sys.executable, '-m', 'pip', 'install', '-q',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu118',
                'torch==2.1.0', 'torchaudio==2.1.0'
            ], check=False, timeout=300)
    except ImportError:
        print("   ⚠️ PyTorch installation may have issues, but continuing...")

# Step 3: Verify PyTorch installation
print("\n3. Verifying PyTorch installation...")
try:
    import torch
    print(f"   ✓ PyTorch {torch.__version__} installed")
    if torch.cuda.is_available():
        print(f"   ✓ CUDA available: {torch.cuda.get_device_name(0)}")
    else:
        print("   ⚠️ CUDA not available (will use CPU)")
except ImportError:
    print("   ❌ PyTorch installation failed")
    raise

# Step 4: Install core dependencies with compatible versions
print("\n4. Installing core dependencies...")
core_packages = [
    'transformers==4.35.2',  # Compatible with PyTorch 2.1
    'numpy==1.24.3',  # Compatible with PyTorch 2.1
    'pandas==2.0.3',
    'scikit-learn==1.3.2',
    'pyyaml==6.0.1',
    'python-dotenv==1.0.0',
    'tqdm==4.66.1',
    'jiwer==3.0.3',
    'tensorboard==2.15.1',
    'wandb==0.16.0',
    'matplotlib==3.8.2',
    'seaborn==0.13.0',
    'pillow==10.2.0',
]

for pkg in core_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

# Step 5: Install audio processing libraries
print("\n5. Installing audio processing libraries...")
audio_packages = [
    'librosa==0.10.1',  # Compatible version
    'soundfile==0.12.1',
    'audioread==3.0.1',
    'pyworld==0.3.4',  # May need system dependencies
]

for pkg in audio_packages:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], 
                      check=False, timeout=120)
    except subprocess.TimeoutExpired:
        print(f"   ⚠️ {pkg} installation timed out, continuing...")

# Step 6: Install NLP libraries
print("\n6. Installing NLP libraries...")
nlp_packages = [
    'underthesea>=1.3.0',  # Flexible version for compatibility
    'datasets==2.16.1',
    'evaluate==0.4.1',
    'pyarrow==14.0.1',
]

for pkg in nlp_packages:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], 
                           check=False, capture_output=True, timeout=180)
    if result.returncode != 0 and 'underthesea' in pkg:
        # underthesea may have issues, try without version constraint
        print(f"   ⚠️ {pkg} failed, trying without version constraint...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'underthesea'], 
                      check=False, timeout=180)

# Step 7: Install database libraries
print("\n7. Installing database libraries...")
db_packages = [
    'sqlalchemy==2.0.23',
    'alembic==1.13.1',
]

for pkg in db_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

# Step 8: Install optional packages (may fail, that's OK)
print("\n8. Installing optional packages...")
optional_packages = [
    'py_vncorenlp',  # May have dependencies
]

for pkg in optional_packages:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], 
                      check=False, timeout=60)
    except:
        print(f"   ⚠️ {pkg} installation skipped (optional)")

# Step 9: Final verification
print("\n9. Verifying installation...")
try:
    import torch
    import torchaudio
    import transformers
    import librosa
    import soundfile
    import numpy
    import pandas
    import sqlalchemy
    
    print("   ✓ All core packages imported successfully")
    
    # Test CUDA
    if torch.cuda.is_available():
        print(f"   ✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"   ✓ CUDA: {torch.version.cuda}")
        print(f"   ✓ cuDNN: {torch.backends.cudnn.version()}")
    else:
        print("   ⚠️ GPU not available")
        
except ImportError as e:
    print(f"   ⚠️ Some packages failed to import: {e}")
    print("   Training may still work, but some features might be unavailable")

print("\n" + "=" * 60)
print("✅ Installation complete!")
print("=" * 60)


### 📁 Accessing Shared Google Drive Folder

If you need to access a shared folder:
1. Open the shared folder link in your browser
2. Right-click the folder → "Add shortcut to Drive"
3. Choose "My Drive" as location
4. The folder will appear in your Drive after mounting

**Shared folder link**: https://drive.google.com/drive/folders/17iqsD7J2xi9_YLQGsqopR8XX-nVFN2y3?usp=sharing


In [ ]:
# 6a) Helper: Auto-detect parquet folder in Drive
import os
from pathlib import Path

def find_parquet_folder():
    """Search for folder containing parquet files in Drive."""
    drive_root = "/content/drive/MyDrive"
    if not os.path.exists(drive_root):
        return None
    
    # Common folder names that might contain the data
    possible_names = [
        "bud500_data",
        "datasets/bud500/data",
        "bud500",
        "17iqsD7J2xi9_YLQGsqopR8XX-nVFN2y3",  # Folder ID
    ]
    
    # Search for folders with parquet files
    found_folders = []
    for root, dirs, files in os.walk(drive_root):
        # Limit search depth to avoid long searches
        depth = root[len(drive_root):].count(os.sep)
        if depth > 3:
            continue
        
        parquet_count = len([f for f in files if f.endswith('.parquet')])
        if parquet_count > 0:
            found_folders.append((root, parquet_count))
    
    return found_folders

print("🔍 Searching for parquet files in your Drive...")
found = find_parquet_folder()

if found:
    print(f"\n✓ Found {len(found)} folder(s) with parquet files:")
    for folder, count in found[:5]:  # Show first 5
        print(f"   📁 {folder} ({count} parquet files)")
    if len(found) > 5:
        print(f"   ... and {len(found) - 5} more")
    print("\n💡 Update BASE_DIR in cell 7 to use one of these paths")
else:
    print("\n⚠️ No parquet files found in Drive")
    print("   Make sure you:")
    print("   1. Mounted Drive (cell 2)")
    print("   2. Added the shared folder to your Drive")
    print("   3. The folder contains .parquet files")


In [ ]:
# 5) Set working directory and environment
import os
import sys

# Change to project directory
if os.path.isdir('/content/ai2text'):
    os.chdir('/content/ai2text')
    print('✓ Changed to project directory:', os.getcwd())
else:
    print('⚠️ Project directory not found at /content/ai2text')
    print('  Current directory:', os.getcwd())

# Set environment variables
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTHONUNBUFFERED'] = '1'  # Unbuffered output for real-time logs

# Add project root to Python path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print('✓ Added project root to Python path')

# Verify key directories exist
required_dirs = ['training', 'models', 'configs', 'database', 'preprocessing']
missing = [d for d in required_dirs if not os.path.isdir(d)]
if missing:
    print('⚠️ Missing directories:', missing)
else:
    print('✓ All required directories found')


In [ ]:
# 6) QUICK TEST config and list parquet shards
import os
import glob
from pathlib import Path

# QUICK TEST MODE
QUICK_TEST = True  # Set to False for full training
MAX_SHARDS_TRAIN = 1 if QUICK_TEST else None
MAX_SHARDS_VAL   = 1 if QUICK_TEST else None
MAX_SHARDS_TEST  = 1 if QUICK_TEST else None
MAX_EXAMPLES_TRAIN = 10 if QUICK_TEST else None
MAX_EXAMPLES_VAL   = 5  if QUICK_TEST else None
MAX_EXAMPLES_TEST  = 5  if QUICK_TEST else None

# ========================================
# AUTO-DETECT ENVIRONMENT AND SET BASE_DIR
# ========================================
# Detect if running locally (Jupyter) or on Colab
IS_COLAB = os.path.exists('/content')
IS_LOCAL = not IS_COLAB

if IS_LOCAL:
    # Local Jupyter environment - use your local Drive path
    BASE_DIR = r"G:\My Drive\datasets\bud500\data"
    print("=" * 60)
    print("🖥️  Running in LOCAL Jupyter environment")
    print("=" * 60)
else:
    # Colab environment - use Drive mount path
    BASE_DIR = "/content/drive/MyDrive/bud500_data"
    print("=" * 60)
    print("☁️  Running in Google Colab")
    print("=" * 60)

print(f"\n📁 Looking for parquet files...")
print(f"   Base directory: {BASE_DIR}")
print("=" * 60)

# Check if directory exists
if not os.path.exists(BASE_DIR):
    print(f"\n❌ Folder not found: {BASE_DIR}")
    if IS_LOCAL:
        print("\n📋 For LOCAL Jupyter:")
        print("   1. Make sure Google Drive is synced locally")
        print("   2. Update BASE_DIR above with your actual local path")
        print("   3. Example: r\"G:\\My Drive\\datasets\\bud500\\data\"")
    else:
        print("\n📋 For Colab:")
        print("   1. Open the shared folder in your browser:")
        print("      https://drive.google.com/drive/folders/17iqsD7J2xi9_YLQGsqopR8XX-nVFN2y3")
        print("   2. Right-click on the folder → 'Add shortcut to Drive'")
        print("   3. Choose 'My Drive' as the location")
        print("   4. Update BASE_DIR above to match the folder name in your Drive")
    raise FileNotFoundError(f"Please configure BASE_DIR correctly. Folder not found: {BASE_DIR}")
else:
    print(f"✓ Folder found: {BASE_DIR}")

train_files = sorted(glob.glob(str(Path(BASE_DIR) / "train-*.parquet")))
val_files   = sorted(glob.glob(str(Path(BASE_DIR) / "validation-*.parquet")))
test_files  = sorted(glob.glob(str(Path(BASE_DIR) / "test-*.parquet")))

if not val_files and not test_files:
    all_files = sorted(glob.glob(str(Path(BASE_DIR) / "*.parquet")))
    val_files = all_files[:1]
    test_files = all_files[1:2]
    train_files = all_files[2:]

def _limit_shards(files, max_n):
    if max_n is not None and len(files) > max_n:
        return files[:max_n], True
    return files, False

train_files, cut_tr = _limit_shards(train_files, MAX_SHARDS_TRAIN)
val_files, cut_va   = _limit_shards(val_files, MAX_SHARDS_VAL)
test_files, cut_te  = _limit_shards(test_files, MAX_SHARDS_TEST)

print(f"Found {len(train_files)} train shards, {len(val_files)} val shards, {len(test_files)} test shards")
if QUICK_TEST:
    if cut_tr: print(f"⚠️ QUICK_TEST: Using only first {MAX_SHARDS_TRAIN} train shards")
    if cut_va: print(f"⚠️ QUICK_TEST: Using only first {MAX_SHARDS_VAL} validation shards")
    if cut_te: print(f"⚠️ QUICK_TEST: Using only first {MAX_SHARDS_TEST} test shards")
    print("🚀 QUICK TEST MODE ENABLED - Training will be fast but limited!")


In [ ]:
# 7) Load datasets from parquet and normalize columns
from datasets import load_dataset, Audio, DatasetDict

# Change these to fit your parquet schema
AUDIO_PATH_COL = "audio_path"   # e.g., audio_path, path, audio_filepath
TEXT_COL       = "text"         # e.g., text, transcript, sentence
TARGET_SR      = 16000

data_files = {}
if train_files: data_files["train"] = train_files
if val_files:   data_files["validation"] = val_files
if test_files:  data_files["test"] = test_files

raw_datasets = load_dataset("parquet", data_files=data_files)

def _limit_examples(ds, max_n):
    if max_n is not None and len(ds) > max_n:
        return ds.select(range(max_n))
    return ds

if QUICK_TEST:
    if "train" in raw_datasets:      raw_datasets["train"]      = _limit_examples(raw_datasets["train"], MAX_EXAMPLES_TRAIN)
    if "validation" in raw_datasets: raw_datasets["validation"] = _limit_examples(raw_datasets["validation"], MAX_EXAMPLES_VAL)
    if "test" in raw_datasets:       raw_datasets["test"]       = _limit_examples(raw_datasets["test"], MAX_EXAMPLES_TEST)

def ensure_audio_column(ds):
    cols = ds.column_names
    # create 'audio' from path if needed
    if "audio" not in cols:
        assert AUDIO_PATH_COL in cols, f"Missing column {AUDIO_PATH_COL}. Please set AUDIO_PATH_COL correctly."
        ds = ds.rename_column(AUDIO_PATH_COL, "audio_path_tmp")
        ds = ds.map(lambda x: {"audio": {"path": x["audio_path_tmp"]}}, remove_columns=["audio_path_tmp"])
    # cast to Audio feature (auto decode/resample)
    ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    return ds

processed = {}
for split in raw_datasets.keys():
    processed[split] = ensure_audio_column(raw_datasets[split])

datasets = DatasetDict(processed)
datasets


In [ ]:
# 8) Export CSV from dataset and copy audio files for DB import
import csv
import shutil
from pathlib import Path

# Check if datasets variable exists
if 'datasets' not in globals():
    print("❌ 'datasets' variable not found!")
    print("  Please run cell 7 first to load datasets")
else:
    EXPORT_BASE = Path('data/external')
    EXPORT_BASE.mkdir(parents=True, exist_ok=True)
    CSV_PATH = EXPORT_BASE / ('parquet_quick.csv' if QUICK_TEST else 'parquet_full.csv')
    
    # Export all splits (train, validation, test) to CSV
    rows = []
    copied_count = 0
    skipped_count = 0
    error_details = []
    
    for split in ["train", "validation", "test"]:
        if split not in datasets:
            print(f"⚠️ Split '{split}' not found in datasets")
            continue
        
        ds = datasets[split]
        print(f"Processing {split} split: {len(ds)} examples")
        
        for idx, ex in enumerate(ds):
            try:
                # Get audio path - datasets library stores it in ex["audio"]
                # After cast_column(Audio), it's a dict with "path" and "array"
                audio_info = ex.get("audio", None)
                if audio_info is None:
                    skipped_count += 1
                    continue
                
                # Handle different audio formats from datasets library
                if isinstance(audio_info, dict):
                    src = audio_info.get("path", None)
                elif hasattr(audio_info, 'get'):
                    src = audio_info.get("path", None)
                else:
                    # Try to get path attribute
                    src = getattr(audio_info, 'path', None)
                
                if not src or not isinstance(src, str):
                    skipped_count += 1
                    if idx < 3:
                        print(f"  Warning: No valid audio path in {split}[{idx}]")
                    continue
                
                # Get transcript text
                txt = ex.get(TEXT_COL, None)
                if txt is None or not str(txt).strip():
                    skipped_count += 1
                    continue
                
                # Copy audio file to EXPORT_BASE
                src_p = Path(src)
                if not src_p.exists():
                    skipped_count += 1
                    if idx < 3:
                        print(f"  Warning: Audio file not found: {src}")
                    continue
                
                # Use original filename, but handle duplicates
                dst = EXPORT_BASE / src_p.name
                if dst.exists() and dst.stat().st_size > 0:
                    # File already copied, reuse it
                    pass
                else:
                    try:
                        shutil.copy2(src, dst)
                        copied_count += 1
                    except Exception as copy_err:
                        skipped_count += 1
                        if idx < 3:
                            print(f"  Error copying file: {copy_err}")
                        continue
                
                # Add to CSV rows
                rows.append({
                    "file_path": dst.relative_to(EXPORT_BASE).as_posix(),
                    "transcript": str(txt).strip()
                })
                
            except Exception as e:
                skipped_count += 1
                error_details.append(f"{split}[{idx}]: {str(e)}")
                if len(error_details) <= 3:  # Only print first few errors
                    print(f"  Error processing {split}[{idx}]: {e}")
    
    # Write CSV
    if rows:
        with open(CSV_PATH, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=["file_path", "transcript"])
            writer.writeheader()
            writer.writerows(rows)
        
        print(f"✓ Wrote CSV: {CSV_PATH}")
        print(f"  Total rows: {len(rows)}")
        print(f"  Files copied: {copied_count}")
        print(f"  Skipped: {skipped_count}")
        if error_details and len(error_details) > 3:
            print(f"  ({len(error_details)} total errors, showing first 3)")
    else:
        print(f"❌ No valid rows to export!")
        print(f"  Check your dataset structure and column names")
        print(f"  Expected columns: audio (with path), {TEXT_COL}")
        if error_details:
            print(f"  Errors encountered: {len(error_details)}")


In [ ]:
# 9) Import CSV into DB with auto-split
import os
csv_path = 'data/external/parquet_quick.csv' if QUICK_TEST else 'data/external/parquet_full.csv'
if os.path.exists(csv_path):
    import subprocess
    result = subprocess.run([
        'python', 'scripts/prepare_data.py',
        '--csv', csv_path,
        '--audio_base', 'data/external',
        '--auto_split', '--skip_duplicates'
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
else:
    print("⚠️ CSV file not found:", csv_path)
    print("Please run cell 8 first to generate the CSV")


In [ ]:
# 10) Pre-flight check: Verify all requirements before training
import os
import sys

print("=" * 60)
print("🔍 PRE-FLIGHT CHECK")
print("=" * 60)

checks_passed = 0
checks_failed = 0

# Check 1: Working directory
print("\n1. Working Directory:")
if os.path.exists('training/train.py'):
    print("   ✓ Project root confirmed")
    print(f"   Current dir: {os.getcwd()}")
    checks_passed += 1
else:
    print("   ❌ Not in project root!")
    print(f"   Current dir: {os.getcwd()}")
    print("   Fix: Run cell 5 to set working directory")
    checks_failed += 1

# Check 2: Required directories
print("\n2. Required Directories:")
required_dirs = ['training', 'models', 'configs', 'database', 'preprocessing']
for d in required_dirs:
    if os.path.isdir(d):
        print(f"   ✓ {d}/")
        checks_passed += 1
    else:
        print(f"   ❌ {d}/ missing")
        checks_failed += 1

# Check 3: Config file
print("\n3. Configuration File:")
config_path = 'configs/colab.yaml'
if os.path.exists(config_path):
    print(f"   ✓ {config_path} exists")
    try:
        import yaml
        with open(config_path, 'r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
        print(f"   ✓ Config loaded successfully")
        print(f"   - Checkpoint dir: {cfg.get('checkpoint_dir', 'N/A')}")
        print(f"   - Database path: {cfg.get('database_path', 'N/A')}")
        print(f"   - Epochs: {cfg.get('num_epochs', 'N/A')}")
        print(f"   - Batch size: {cfg.get('batch_size', 'N/A')}")
        checks_passed += 1
    except Exception as e:
        print(f"   ❌ Error loading config: {e}")
        checks_failed += 1
else:
    print(f"   ❌ {config_path} not found")
    print("   Fix: Run cell 10 to generate config")
    checks_failed += 1

# Check 4: Database
print("\n4. Database:")
if 'cfg' in locals():
    db_path = cfg.get('database_path', 'database/asr_training.db')
    if not os.path.isabs(db_path):
        db_path = os.path.join(os.getcwd(), db_path)
    if os.path.exists(db_path):
        size_mb = os.path.getsize(db_path) / (1024 * 1024)
        print(f"   ✓ Database exists: {db_path}")
        print(f"   Size: {size_mb:.2f} MB")
        checks_passed += 1
    else:
        print(f"   ❌ Database not found: {db_path}")
        print("   Fix: Run cell 9 to import data")
        checks_failed += 1

# Check 5: Training script
print("\n5. Training Script:")
if os.path.exists('training/train.py'):
    print("   ✓ training/train.py exists")
    checks_passed += 1
else:
    print("   ❌ training/train.py not found")
    checks_failed += 1

# Check 6: Python path
print("\n6. Python Environment:")
print(f"   Python: {sys.executable}")
print(f"   Version: {sys.version.split()[0]}")
project_root = os.getcwd()
if project_root in sys.path:
    print(f"   ✓ Project root in Python path")
    checks_passed += 1
else:
    print(f"   ⚠️ Project root not in Python path (may still work)")
    checks_passed += 1

# Check 7: GPU availability
print("\n7. GPU Check:")
try:
    import torch
    if torch.cuda.is_available():
        print(f"   ✓ CUDA available")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        print(f"   CUDA version: {torch.version.cuda}")
        checks_passed += 1
    else:
        print("   ⚠️ CUDA not available (will use CPU - slower)")
        checks_passed += 1
except ImportError:
    print("   ⚠️ PyTorch not installed")
    checks_failed += 1

# Summary
print("\n" + "=" * 60)
print("📊 SUMMARY")
print("=" * 60)
print(f"✅ Passed: {checks_passed}")
print(f"❌ Failed: {checks_failed}")

if checks_failed == 0:
    print("\n🎉 All checks passed! Ready to train.")
    print("   → Proceed to cell 11 to start training")
else:
    print(f"\n⚠️ {checks_failed} check(s) failed. Please fix before training.")
print("=" * 60)


In [ ]:
# 10) Configure checkpoint directory on Google Drive and write a Colab config
import os
import yaml
from datetime import datetime

# Ensure we're in the project directory
if not os.path.exists('configs/default.yaml'):
    print('⚠️ configs/default.yaml not found. Make sure you are in the project root.')
    print('  Current directory:', os.getcwd())

# Setup checkpoint directory on Drive
drive_base = '/content/drive/MyDrive/asr_checkpoints'
os.makedirs(drive_base, exist_ok=True)
run_name = datetime.now().strftime('%Y%m%d-%H%M%S')
checkpoint_dir = f'{drive_base}/{run_name}'
os.makedirs(checkpoint_dir, exist_ok=True)

# Load default config
try:
    with open('configs/default.yaml', 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
except Exception as e:
    print(f'❌ Error loading config: {e}')
    raise

# Update checkpoint directory to Drive (absolute path)
cfg['checkpoint_dir'] = checkpoint_dir

# Ensure database path is correct (relative to project root)
if 'database_path' not in cfg:
    cfg['database_path'] = 'database/asr_training.db'
    
# Verify database exists
db_path = cfg['database_path']
if not os.path.isabs(db_path):
    db_path = os.path.join(os.getcwd(), db_path)
if not os.path.exists(db_path):
    print(f'⚠️ Database not found at {db_path}')
    print('  Make sure you ran cell 9 to import data first!')
else:
    print(f'✓ Database found: {db_path}')

# For QUICK_TEST mode, reduce epochs and batch size
if QUICK_TEST:
    cfg['num_epochs'] = 1
    cfg['batch_size'] = 4
    print("⚠️ QUICK_TEST: Setting num_epochs=1, batch_size=4")

# Save Colab-specific config
with open('configs/colab.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print('✓ Colab config written to configs/colab.yaml')
print('✓ Checkpoint dir ->', checkpoint_dir)
print('✓ Database path ->', cfg['database_path'])


In [ ]:
# 11) Quick training (1 epoch) or full training (saving to Drive)
import os
import subprocess
import sys

# Ensure we're in the project directory
if not os.path.exists('training/train.py'):
    print('❌ training/train.py not found!')
    print('  Current directory:', os.getcwd())
    print('  Please run cell 5 first to set working directory')
else:
    # Verify config exists
    config_path = 'configs/colab.yaml'
    if not os.path.exists(config_path):
        print(f'❌ Config file not found: {config_path}')
        print('  Please run cell 11 first to generate config')
    else:
        # Verify database exists
        import yaml
        with open(config_path, 'r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
        db_path = cfg.get('database_path', 'database/asr_training.db')
        if not os.path.isabs(db_path):
            db_path = os.path.join(os.getcwd(), db_path)
        
        if not os.path.exists(db_path):
            print(f'❌ Database not found: {db_path}')
            print('  Please run cell 9 first to import data')
            print('  Or run cell 10 to check all requirements')
        else:
            print("🚀 Starting training...")
            print(f"  Config: {config_path}")
            print(f"  Database: {db_path}")
            print(f"  Checkpoint dir: {cfg.get('checkpoint_dir', 'N/A')}")
            print(f"  Epochs: {cfg.get('num_epochs', 'N/A')}")
            print(f"  Batch size: {cfg.get('batch_size', 'N/A')}")
            print("-" * 60)
            
            # Run training with real-time output
            try:
                result = subprocess.run(
                    [sys.executable, 'training/train.py', '--config', config_path],
                    cwd=os.getcwd(),
                    env=dict(os.environ, PYTHONUNBUFFERED='1'),
                    check=False  # Don't raise on non-zero exit
                )
                
                print("-" * 60)
                if result.returncode == 0:
                    print("✅ Training completed successfully!")
                else:
                    print(f"❌ Training failed with exit code: {result.returncode}")
                    print("  Check the error messages above for details")
            except Exception as e:
                print(f"❌ Error running training: {e}")
                import traceback
                traceback.print_exc()


In [ ]:
# 12) Optional: Evaluate a checkpoint (on Drive)
import glob
import os
import subprocess
import sys

# Find latest checkpoint
checkpoint_pattern = '/content/drive/MyDrive/asr_checkpoints/*/last.ckpt'
checkpoints = glob.glob(checkpoint_pattern)
if checkpoints:
    latest_checkpoint = sorted(checkpoints, key=os.path.getmtime, reverse=True)[0]
    print("📊 Evaluating checkpoint:", latest_checkpoint)
    
    result = subprocess.run([
        sys.executable, 'training/evaluate.py',
        '--config', 'configs/colab.yaml',
        '--checkpoint', latest_checkpoint,
        '--split', 'test'
    ], capture_output=False, text=True)
else:
    print("⚠️ No checkpoints found. Train the model first (cell 12).")


## 📝 Summary

### Complete Training Pipeline:
1. ✅ **Cell 1**: Check GPU availability
2. ✅ **Cell 2**: Mount Google Drive
3. ✅ **Cell 3**: Clone project from GitHub
4. ✅ **Cell 4**: Install dependencies
5. ✅ **Cell 5**: Set working directory and environment
6. ✅ **Cell 6**: Configure parquet paths and QUICK_TEST mode
7. ✅ **Cell 7**: Load datasets from parquet files
8. ✅ **Cell 8**: Export CSV and copy audio files
9. ✅ **Cell 9**: Import data to database
10. ✅ **Cell 10**: Pre-flight check (verify all requirements)
11. ✅ **Cell 11**: Configure checkpoint directory on Drive
12. ✅ **Cell 12**: Start training (checkpoints saved to Drive)
13. ✅ **Cell 13**: Evaluate trained model (optional)

### Important Notes:
- **Run cells in order** - Each cell depends on previous ones
- **Check Cell 10** before training - It verifies everything is ready
- **Checkpoints Location**: `/content/drive/MyDrive/asr_checkpoints/<timestamp>/`
- **QUICK_TEST Mode**: Set `QUICK_TEST = False` in Cell 6 for full training

### Troubleshooting:
- If training fails, check Cell 10 output for missing requirements
- Ensure database exists (Cell 9 must complete successfully)
- Verify config file exists (Cell 11 must run)
- Check GPU is available (Cell 1 should show GPU info)
